# Timing (20 pairs), epsilon sweep, rho*, block shuffle, numbers autofill

Cell 3: the separate 20-pair timing protocol that `\VacherWSec` actually comes from (not the 120-pair `tab:runtime` protocol -- see INVENTORY.md section 0 item 7). Cell 5: the entropy/variance epsilon sweep (`\EntropyLow/High`, `\VarianceLow/High`, `\TargetGap`). Cell 10: plan-alignment rho* (`\RhoHealthy`, `\RhoFlagged`, `fig:app-rho`).

**Needs:** DTD, the CNT checkpoint, the Vacher baseline repo (`install/vacher.sh`) for cell 3's Vacher timing.

**Maintained runnable path:** none of these three sub-studies has a dedicated experiments/ script yet; this notebook (and its cached outputs in `data/paper_completion/`, already rescored by `score_only.py`) is the only record.

# Paper completion run — everything missing, one session
Produces, in order: (0) pipeline constants → Method `[MISSING]`s; (1) timing rerun; (2) KID from saved images;
(3) ε sweep + endpoint gap; (4) ρ* over the saved n=1000 pairs; (5) block-shuffle score; (6) all figures; (7) `numbers_autofill.tex`.

**Resumable.** Every block checks Drive for its own output before running. `OUT = DATA_ROOT/paper_completion/`.
**Seeded.** Every pair uses `seed = hash(k)`; all new numbers are deterministic. (The earlier tables were produced unseeded — say so in the paper.)
**Stubs you must fill:** `vacher_midpoint()` in block 1 (paste from your Vacher notebook) and `FACE_A/FACE_B` paths in block 6.

In [ ]:
# ============================================================
# 0. SETUP — identical pipeline to tm_scale1000 (CELL 4), plus a plan-returning Sinkhorn and per-pair seeding.
# ============================================================
!pip install -q pot plenoptic einops timm lpips
import os, sys, glob, random, time, gc, math, json
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile
from scipy.stats import wilcoxon
from google.colab import drive
drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES = True
device = 'cuda' if torch.cuda.is_available() else 'cpu'

BASE = os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
OUT  = f'{BASE}/paper_completion'; os.makedirs(OUT, exist_ok=True)
FIGS = f'{OUT}/figs'; os.makedirs(FIGS, exist_ok=True)
SCALE = f'{BASE}/tm_scale1000'          # saved n=1000 midpoints + manifest
TM50  = f'{BASE}/tm_comparison'         # saved n=50 run
TMP = '/content/tmp_imgs'; os.makedirs(TMP, exist_ok=True)
NUM = {}                                # macro name -> value, dumped at the end
def note(k, v): NUM[k] = v; print(f'  {k} = {v}')

LOCAL_DTD = '/content/dtd_images'
if len(glob.glob(f'{LOCAL_DTD}/*/*.jpg')) < 5000:
    os.system(f'cp -r "{BASE}/dtd/images" {LOCAL_DTD}')
DTD_ROOT = LOCAL_DTD
cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def dtd_path(c,s): return random.Random(s).choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))
print(f'{len(cats)} categories')

os.chdir('/content'); os.system('rm -rf cnt-siga24')
os.system('git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24')
os.system('pip install -q -r cnt-siga24/requirements.in')
os.system('pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT_DIR='/content/cnt-siga24'; os.chdir(CNT_DIR); sys.path.insert(0,CNT_DIR)
from src.inference.core import TexAutoEncoderInference, load_image
model = TexAutoEncoderInference(ckpt_path=Path(f"{CNT_DIR}/ckpts/finetuned-model256.ckpt"), device=device)
IMG_SIZE = 256
FMS = [round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]
EPS, ITERS, NSUB = 0.05, 100, 4096

_tc=[0]
def cnt_load(x, size=IMG_SIZE):
    if isinstance(x,(str,Path)): return load_image(Path(x), size).to(device)
    _tc[0]+=1; fp=f'{TMP}/t{_tc[0]%64}.png'; x.convert('RGB').save(fp)
    return load_image(Path(fp), size).to(device)
def to_pil(dec):
    a=((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy()
    return Image.fromarray((a*255).astype(np.uint8))
def get_gen_inputs(img):
    with torch.no_grad():
        blob=model.encode(img); L=model.splat(blob)
        return blob, L["layer_level_layouts"][-1][0]["blob_features"].clone(), L["layer_level_layouts"][-1][0]["score_img"].clone()
def decode_interp(blob_ref, bf, si):
    with torch.no_grad():
        L=model.splat(blob_ref)
        L["layer_level_layouts"][-1][0]["blob_features"]=bf; L["layer_level_layouts"][-1][0]["score_img"]=si
        return model.decode(L)
def build_style_maps(bf,si,fms):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(fms):
        s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref, sm):
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]
        sp={}; s=si; sp[s.shape[-1]]=s
        while s.shape[-1]>min(fms):
            s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1)
        x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)

# --- Sinkhorn that returns the plan (needed for entropy, rho*, variance) ---
def sinkhorn_plan(src,tgt,eps=EPS,n_iters=ITERS):
    N,M=src.shape[0],tgt.shape[0]; C=torch.cdist(src,tgt).pow(2); K=torch.exp(-C/eps)
    a=torch.ones(N,1,device=src.device)/N; b=torch.ones(M,1,device=src.device)/M; u=torch.ones(N,1,device=src.device)
    for _ in range(n_iters): v=b/(K.T@u+1e-10); u=a/(K@v+1e-10)
    P=u*K*v.T; return P, C
def sinkhorn_transport(src,tgt,eps=EPS,n_iters=ITERS):
    P,_=sinkhorn_plan(src,tgt,eps,n_iters); return (P/(P.sum(1,keepdim=True)+1e-10))@tgt

def build_d(ia, ib):
    ba,bfa,sia=get_gen_inputs(ia); bb,bfb,sib=get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def _grids(d):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sa[L][0],sb[L][0]; C,H,W=A.shape
    return L,C,H,W, A.permute(1,2,0).reshape(-1,C), B.permute(1,2,0).reshape(-1,C)
_TAG={'s':1,'t':2,'a':3,'b':4}
def _sub(N, seed, tag):
    g=torch.Generator(device=device).manual_seed(int(seed)*7919+_TAG[tag])
    return torch.randperm(N,generator=g,device=device)[:NSUB]
def _dec(d, it, L, H, W, C, layout='a'):
    it=it.reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    return decode_with_style_maps(d["blob_"+layout], {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)})

def interp_ot(d, eta, seed=0, eps=EPS, return_plan=False):
    L,C,H,W,sf,tf=_grids(d); N=sf.shape[0]
    i_s=_sub(N,seed,'s'); i_t=_sub(N,seed,'t')
    with torch.no_grad(): P,Cm=sinkhorn_plan(sf[i_s],tf[i_t],eps); tt=(P/(P.sum(1,keepdim=True)+1e-10))@tf[i_t]
    nn=torch.cdist(sf,sf[i_s]).argmin(1)
    out=_dec(d,(1-eta)*sf+eta*tt[nn],L,H,W,C)
    return (out,dict(P=P,C=Cm,src=sf[i_s],tgt=tf[i_t],tt=tt,nn=nn,sf=sf,tf=tf)) if return_plan else out
def interp_ot_sym(d, eta, seed=0):
    # AS IMPLEMENTED: the A-grid and B-grid interpolants are averaged ELEMENTWISE AT THE SAME GRID POSITION
    # and decoded through A's layout. Write this honestly in the Method.
    L,C,H,W,af,bf=_grids(d); N=af.shape[0]
    ia=_sub(N,seed,'a'); ib=_sub(N,seed,'b')
    with torch.no_grad(): tab=sinkhorn_transport(af[ia],bf[ib]); tba=sinkhorn_transport(bf[ib],af[ia])
    na=torch.cdist(af,af[ia]).argmin(1); nb=torch.cdist(bf,bf[ib]).argmin(1)
    return _dec(d,0.5*(((1-eta)*af+eta*tab[na])+(eta*bf+(1-eta)*tba[nb])),L,H,W,C)
def interp_lin_pixel(d, eta, seed=0):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"], {s:(1-eta)*sa[s]+eta*sb[s] for s in set(FMS)})
def interp_lin_texton(d, eta, seed=0):
    return decode_interp(d["blob_a"], (1-eta)*d["bf_a"]+eta*d["bf_b"], (1-eta)*d["si_a"]+eta*d["si_b"])
FNS = {'pixel_linear':interp_lin_pixel,'texton_linear':interp_lin_texton,'ot':interp_ot,'ot_sym':interp_ot_sym}
def make_recon(X_pil):
    e=cnt_load(X_pil); return FNS['pixel_linear'](build_d(e,e),0.0)

# --- observers ---
import timm, lpips
dino = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0).to(device).eval()
for p in dino.parameters(): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1); _S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
@torch.no_grad()
def dino_feats(pils, bs=16):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device); b=F.interpolate(b,size=518,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(dino(b).cpu().numpy())
    return np.concatenate(out)
from torchvision.models import inception_v3, Inception_V3_Weights
_inc = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True).to(device).eval(); _inc.fc=torch.nn.Identity()
for p in _inc.parameters(): p.requires_grad_(False)
@torch.no_grad()
def inc_feats(pils, bs=24):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device); b=F.interpolate(b,size=299,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(_inc(b).cpu().numpy())
    return np.concatenate(out)
lp = lpips.LPIPS(net='alex').to(device).eval()
@torch.no_grad()
def lpips_d(p1,p2):
    t=lambda p: (torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/127.5-1)[None].to(device)
    return float(lp(t(p1),t(p2)))
print('ready')

In [ ]:
# ============================================================
# 0b. PIPELINE CONSTANTS  → fills Method [MISSING]s
# ============================================================
A0 = cnt_load(dtd_path('cracked', 1)); B0 = cnt_load(dtd_path('lined', 1))
d0 = build_d(A0, B0); L,C,H,W,sf,tf = _grids(d0)
note('StyleMapGrid', f'{H}x{W}'); note('StyleMapN', H*W); note('FeatureDim', C); note('FMS', str(FMS))
note('NumBlobs', d0['bf_a'].shape[1]); note('BlobFeatDim', d0['bf_a'].shape[2]); note('ScoreImgShape', str(tuple(d0['si_a'].shape)))
note('SinkhornEpsAbsolute', EPS); note('SinkhornIters', ITERS); note('SubsampleM', NSUB)
_, info = interp_ot(d0, 0.5, seed=1, return_plan=True)
Cmed = float(info['C'].median()); note('EpsOverMedianCost', EPS/Cmed); note('MedianCost', Cmed)
used = len(torch.unique(info['nn'])); note('NNExtensionSampledUsed', f'{used} of {NSUB}')
# variance of NN-extended cloud vs subsample-only: the "expansion costs <10% of transported variance" check
tt_full = info['tt'][info['nn']]
note('ExtensionVarRatio', float(tt_full.var(0).sum()/info['tt'].var(0).sum()))
print('feature norm stats: mean', float(sf.norm(dim=1).mean()), ' centred?', float(sf.mean(0).norm()))

In [ ]:
# ============================================================
# 1. TIMING RERUN — all seven methods, one A100, one resolution, encode split from interpolate+decode.
#    Same timed() discipline as compute_cost_5_4: warm-up, cuda sync, median of reps; Vacher reps=1 (each call is a full optimisation).
# ============================================================
import gc
os.chdir('/content'); os.system('rm -rf texture-interpolation')
os.system('git clone -q --depth 1 https://github.com/JonathanVacher/texture-interpolation.git')
VACHER_DIR = '/content/texture-interpolation/texture-synthesis-algorithm'; os.chdir(CNT_DIR)
print('Vacher models:', os.listdir(f'{VACHER_DIR}/models'))
TMPV='/content/vacher_tmp'; os.makedirs(TMPV,exist_ok=True)
def prep_square(path,out,size=IMG_SIZE):
    Image.open(path).convert('RGB').resize((size,)*2,Image.BILINEAR).save(out); return out

def vacher_interp(pathA, pathB, weight=0.5, model_name='tex_wasser', n_iter=30, bfgs_iter=20, shape='256,256', seed=0):
    cwd=os.getcwd(); os.chdir(VACHER_DIR)
    if VACHER_DIR not in sys.path: sys.path.insert(0,VACHER_DIR)
    for mod in [m for m in list(sys.modules) if m.startswith(('models','options'))]: del sys.modules[mod]
    from options.base_options import BaseOptions
    from models import create_model, utils
    argv_bak=sys.argv
    sys.argv=['run_synthesis.py','--input_tex_1',str(pathA),'--input_tex_2',str(pathB),'--weight_1',str(weight),'--model',model_name,
              '--n_iter',str(n_iter),'--bfgs_iter',str(bfgs_iter),'--shape',shape,'--seed',str(seed),'--display_freq','100000']
    try:
        opt=BaseOptions().parse(); m=create_model(opt); m.get_net_and_loss()
        for _ in range(opt.n_iter+1): m.optimize_parameters()
        arr=utils.tensor2im(m.output_tex.data.clamp_(0,1))
        return Image.fromarray(arr.astype(np.uint8)).convert('RGB').resize((IMG_SIZE,)*2)
    finally:
        sys.argv=argv_bak; os.chdir(cwd)

def timed(fn, warmup=1, reps=5):
    for _ in range(warmup): fn()
    torch.cuda.synchronize(); gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    ts=[]
    for _ in range(reps):
        torch.cuda.synchronize(); t0=time.perf_counter(); out=fn(); torch.cuda.synchronize(); ts.append(time.perf_counter()-t0)
    return float(np.median(ts)), torch.cuda.max_memory_allocated()/2**20, out

VAR=['tex_gram','tex_gauss','tex_wasser']; N_TIME=20; K_PATH=11; ETAS=[i/(K_PATH-1) for i in range(K_PATH)]
TIMCSV=f'{OUT}/timing_n{N_TIME}.csv'
if os.path.exists(TIMCSV):
    dft=pd.read_csv(TIMCSV); print('timing loaded')
else:
    rngt=random.Random(555); pairs_t=[]
    for i in range(N_TIME):
        a,b=rngt.sample(cats,2); pairs_t.append((dtd_path(a,60000+i),dtd_path(b,61000+i)))
    rows=[]
    for k,(pa,pb) in enumerate(pairs_t):
        qa,qb=prep_square(pa,f'{TMPV}/A.png'),prep_square(pb,f'{TMPV}/B.png')
        t_enc,_,_=timed(lambda: build_d(cnt_load(qa),cnt_load(qb)))
        d=build_d(cnt_load(qa),cnt_load(qb)); rec=dict(pair=k,encode_s=t_enc)
        for m,fn in FNS.items():
            t,mem,_=timed(lambda: fn(d,0.5,seed=k)); rec[f'{m}_interp_s']=t; rec[f'{m}_mem_MB']=mem
            tp,_,_=timed(lambda: [fn(d,e,seed=k) for e in ETAS], warmup=0, reps=1); rec[f'{m}_path_interp_s']=tp
        for vm in VAR:
            try:
                t,mem,_=timed(lambda: vacher_interp(qa,qb,0.5,vm,n_iter=30), warmup=0, reps=1)
                rec[f'{vm}_interp_s']=t; rec[f'{vm}_mem_MB']=mem
            except Exception as e:
                print(f'  {vm} failed on pair {k}: {type(e).__name__}: {str(e)[:100]}'); rec[f'{vm}_interp_s']=np.nan
        rows.append(rec); pd.DataFrame(rows).to_csv(TIMCSV,index=False); print(f'  {k+1}/{N_TIME}')
    dft=pd.DataFrame(rows)

E=float(dft.encode_s.median()); note('EncodeSec',round(E,3))
print(f"\n{'method':14}{'interp/mid':>12}{'total/mid':>11}{'path11':>9}{'peak MB':>9}")
for m in FNS:
    mid=float(dft[f'{m}_interp_s'].median()); path=E+float(dft[f'{m}_path_interp_s'].median())
    print(f"{m:14}{mid:12.3f}{E+mid:11.3f}{path:9.2f}{dft[f'{m}_mem_MB'].median():9.0f}")
    note(f'Sec_{m}_midpoint_total',round(E+mid,3)); note(f'Sec_{m}_path11_total',round(path,2)); note(f'Sec_{m}_interp_only',round(mid,3))
for vm in VAR:
    if dft[f'{vm}_interp_s'].notna().any():
        mid=float(dft[f'{vm}_interp_s'].median()); path=K_PATH*mid
        print(f"{vm:14}{mid:12.2f}{mid:11.2f}{path:9.1f}{dft[f'{vm}_mem_MB'].median():9.0f}")
        note(f'Sec_{vm}_midpoint',round(mid,1)); note(f'Sec_{vm}_path11',round(path,1))
if dft['tex_wasser_interp_s'].notna().any():
    w=float(dft['tex_wasser_interp_s'].median())
    note('SpeedupW_midpoint',round(w/(E+float(dft['ot_interp_s'].median()))))
    note('SpeedupW_path',round(K_PATH*w/(E+float(dft['ot_path_interp_s'].median()))))
    note('SpeedupGr_midpoint',round(float(dft['tex_gram_interp_s'].median())/(E+float(dft['ot_interp_s'].median()))))

# resolution scaling (Vacher only; CNT is fixed at 256)
RS=f'{OUT}/res_scaling.json'
if not os.path.exists(RS):
    pa,pb=pairs_t[0] if 'pairs_t' in dir() else (dtd_path(cats[3],11),dtd_path(cats[17],22)); rs={}
    for size in [128,256,512]:
        qa=prep_square(pa,f'{TMPV}/A{size}.png',size); qb=prep_square(pb,f'{TMPV}/B{size}.png',size)
        try: t,_,_=timed(lambda: vacher_interp(qa,qb,0.5,'tex_wasser',n_iter=30,shape=f'{size},{size}'),warmup=0,reps=1); rs[size]=t
        except Exception as e: rs[size]=None
    json.dump(rs,open(RS,'w'))
print('Vacher W2 by resolution:', json.load(open(RS)))

In [ ]:
# ============================================================
# 2. KID from SAVED images (n=1000 sets on Drive; n=50 sets if present). Polynomial-kernel MMD, unbiased.
# ============================================================
def kid(X, Y, n_sub=1000, n_rep=20, seed=0):
    rng=np.random.RandomState(seed); d=X.shape[1]; m=min(n_sub,len(X),len(Y)); vals=[]
    for _ in range(n_rep):
        x=X[rng.choice(len(X),m,replace=False)]; y=Y[rng.choice(len(Y),m,replace=False)]
        k=lambda a,b: (a@b.T/d+1)**3
        kxx,kyy,kxy=k(x,x),k(y,y),k(x,y)
        vals.append((kxx.sum()-np.trace(kxx))/(m*(m-1)) + (kyy.sum()-np.trace(kyy))/(m*(m-1)) - 2*kxy.mean())
    return float(np.mean(vals)), float(np.std(vals))
def feats_dir(p, n=None):
    fs=sorted(glob.glob(f'{p}/*.png'))[:n]; return inc_feats([Image.open(f) for f in fs]) if fs else None
REF = f'{OUT}/ref_inc_feats.npy'
if os.path.exists(REF): ref=np.load(REF)
else:
    # same 2,819-image reference as the FID runs if a list exists; else rebuild from the same seed rule
    lst=f'{SCALE}/fid_reference_list.json'
    if os.path.exists(lst): files=json.load(open(lst))
    else:
        random.seed(2819); files=[]
        for c in cats: files+=random.sample(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')),60)
        files=files[:2819]; json.dump(files,open(f'{OUT}/fid_reference_list_REBUILT.json','w'))
        print('WARNING: reference rebuilt — confirm it matches the FID runs')
    ref=inc_feats([Image.open(f).convert('RGB').resize((128,128)) for f in files]); np.save(REF,ref)
note('FIDRefN', len(ref))
K={}
for s in ['pixel_linear','texton_linear','ot','ot_sym','mid','recon_A']:
    p=f'{SCALE}/{s}'
    if os.path.isdir(p):
        X=feats_dir(p); K[f'k1000_{s}']=kid(X,ref); K[f'k50_{s}']=kid(X[:50],ref,n_sub=50)
        note(f'KID1000_{s}', '%.4f±%.4f'%K[f'k1000_{s}']); note(f'KID50_{s}', '%.4f±%.4f'%K[f'k50_{s}'])
json.dump(K,open(f'{OUT}/kid.json','w'),indent=1)

In [ ]:
# ============================================================
# 3. ε SWEEP + ENDPOINT GAP  (20 pairs; entropy, variance ratio, decoded LPIPS/DINO delta; rho* per eps)
# ============================================================
SW=f'{OUT}/eps_sweep.json'
if os.path.exists(SW): sweep=json.load(open(SW))
else:
    random.seed(23); pairs=[]
    while len(pairs)<20:
        ca,cb=random.sample(cats,2); pairs.append((dtd_path(ca,len(pairs)+300), dtd_path(cb,len(pairs)+400)))
    EPSS=[0.005,0.01,0.02,0.05,0.1,0.2,0.5]; rows=[]
    for k,(pa,pb) in enumerate(pairs):
        d=build_d(cnt_load(pa),cnt_load(pb)); decs={}
        for e in EPSS:
            out,info=interp_ot(d,0.5,seed=k,eps=e,return_plan=True); P=info['P']
            Pr=P/(P.sum(1,keepdim=True)+1e-10); H_row=float((-(Pr*torch.log(Pr+1e-12)).sum(1)).mean())
            var_ratio=float(info['tt'].var(0).sum()/info['tgt'].var(0).sum())
            xs=info['src']-info['src'].mean(0); ys=info['tgt']-info['tgt'].mean(0)
            rho=float((P*(xs@ys.T)).sum()/(xs.norm(dim=1).pow(2).mean().sqrt()*ys.norm(dim=1).pow(2).mean().sqrt()))
            decs[e]=to_pil(out); rows.append(dict(k=k,eps=e,row_entropy=H_row,var_ratio=var_ratio,rho=rho))
        # decoded change between the two ends of the sweep, and vs default
        rows[-1]['lpips_ends']=lpips_d(decs[EPSS[0]],decs[EPSS[-1]]); rows[-1]['dino_ends']=float(np.linalg.norm(np.diff(dino_feats([decs[EPSS[0]],decs[EPSS[-1]]]),axis=0)))
        # endpoint gap: decoded eta=1 vs recon(B), normalised by |A-B| in DINO
        e1=to_pil(interp_ot(d,1.0,seed=k)); rA=to_pil(make_recon(Image.open(pa))); rB=to_pil(make_recon(Image.open(pb)))
        f=dino_feats([e1,rA,rB]); rows[-1]['endpoint_gap']=float(np.linalg.norm(f[0]-f[2])/np.linalg.norm(f[1]-f[2]))
        if k==0:
            fig,ax=plt.subplots(1,len(EPSS),figsize=(3*len(EPSS),3))
            for a,e in zip(ax,EPSS): a.imshow(decs[e]); a.set_title(f'eps={e}'); a.axis('off')
            plt.savefig(f'{FIGS}/eps_sweep_example.png',dpi=150,bbox_inches='tight'); plt.show()
    sweep=rows; json.dump(sweep,open(SW,'w'),indent=1)
df=pd.DataFrame(sweep); g=df.groupby('eps')[['row_entropy','var_ratio','rho']].median(); print(g)
note('RowEntropyCeil', round(math.log(NSUB),2))
note('EpsEntropyHigh', round(g.loc[0.5,'row_entropy'],2)); note('EpsEntropyLow', round(g.loc[0.005,'row_entropy'],2))
note('EpsEntropyDefault', round(g.loc[0.05,'row_entropy'],2))
note('EpsVarLow', f"{100*g.loc[0.5,'var_ratio']:.0f}\\%"); note('EpsVarHigh', f"{100*g.loc[0.005,'var_ratio']:.0f}\\%"); note('EpsVarDefault', f"{100*g.loc[0.05,'var_ratio']:.0f}\\%")
ends=df.dropna(subset=['lpips_ends'])
note('EpsDecodeDeltaLPIPS', f"{ends.lpips_ends.median():.3f} (IQR {ends.lpips_ends.quantile(.25):.3f}--{ends.lpips_ends.quantile(.75):.3f})")
note('EpsDecodeDeltaDINO', f"{ends.dino_ends.median():.1f}")
note('EndpointGap', round(float(ends.endpoint_gap.median()),3))
note('RhoStarDefaultMedian', round(g.loc[0.05,'rho'],3))

In [ ]:
# ============================================================
# 4. rho* OVER THE SAVED n=1000 PAIRS  → N screened, cluster separation, threshold set by 2-means (report honestly)
# ============================================================
RH=f'{OUT}/rhostar_1000.csv'
if os.path.exists(RH): rh=pd.read_csv(RH)
else:
    MAN=json.load(open(f'{SCALE}/manifest.json')); rows=[]
    for p_ in MAN['pairs']:
        k=p_['k']; A=Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'); B=Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB')
        d=build_d(cnt_load(A),cnt_load(B)); _,info=interp_ot(d,0.5,seed=k,return_plan=True); P=info['P']
        xs=info['src']-info['src'].mean(0); ys=info['tgt']-info['tgt'].mean(0)
        rho=float((P*(xs@ys.T)).sum()/(xs.norm(dim=1).pow(2).mean().sqrt()*ys.norm(dim=1).pow(2).mean().sqrt()))
        rows.append(dict(k=k,rho=rho,catA=p_.get('catA'),catB=p_.get('catB')))
        if k%100==0: print(k, rho)
    rh=pd.DataFrame(rows); rh.to_csv(RH,index=False)
from sklearn.cluster import KMeans
km=KMeans(2,n_init=10,random_state=0).fit(rh[['rho']]); rh['flag']=km.labels_==np.argmin(km.cluster_centers_.ravel())
hi=rh[~rh.flag].rho; lo=rh[rh.flag].rho
note('RhoStarNScreened', len(rh)); note('RhoStarHealthy', f'{hi.mean():.3f} \\pm {hi.std():.3f}')
note('RhoStarFail', f'{lo.mean():.2f}' if len(lo) else 'none'); note('RhoStarNFlagged', int(rh.flag.sum()))
note('RhoStarGap', f'min healthy {hi.min():.3f} / max flagged {lo.max():.3f}' if len(lo) else 'n/a')
plt.hist(rh.rho,bins=60); plt.xlabel(r'$\rho^\star$'); plt.savefig(f'{FIGS}/rhostar_hist.png',dpi=150,bbox_inches='tight'); plt.show()
print(rh[rh.flag])

In [ ]:
# ============================================================
# 5. BLOCK-SHUFFLE SCORE — does the marginal carry identity? (32 textures: 16 stochastic, 16 regular categories)
# ============================================================
STOCH=['cracked','marbled','fibrous','bubbly','porous','scaly','stained','veined','blotchy','frilly','sprinkled','pitted','crystalline','flecked','matted','smeared']
REG=['honeycombed','grid','striped','banded','chequered','lined','polka-dotted','woven','zigzagged','lacelike','crosshatched','dotted','meshed','braided','waffled','paisley']
def block_shuffle_score(pil, block=4, seed=0):
    e=cnt_load(pil); d=build_d(e,e); L,C,H,W,sf,_=_grids(d)
    g=torch.Generator(device=device).manual_seed(seed)
    grid=sf.reshape(H,W,C); hb,wb=H//block,W//block
    blocks=grid[:hb*block,:wb*block].reshape(hb,block,wb,block,C).permute(0,2,1,3,4).reshape(hb*wb,block,block,C)
    perm=torch.randperm(hb*wb,generator=g,device=device); shuf=blocks[perm].reshape(hb,wb,block,block,C).permute(0,2,1,3,4).reshape(hb*block,wb*block,C)
    full=grid.clone(); full[:hb*block,:wb*block]=shuf
    rec=to_pil(_dec(d,grid.reshape(-1,C),L,H,W,C)); shf=to_pil(_dec(d,full.reshape(-1,C),L,H,W,C))
    f=dino_feats([rec,shf,pil]); return float(np.linalg.norm(f[0]-f[1])/np.linalg.norm(f[0]-f[2]+1e-9)), rec, shf
BS=f'{OUT}/block_shuffle.json'
if os.path.exists(BS): bs=json.load(open(BS))
else:
    bs=[]
    for grp,cl in [('stochastic',STOCH),('regular',REG)]:
        for c in cl:
            if c not in cats: continue
            s,rec,shf=block_shuffle_score(Image.open(dtd_path(c,5)).convert('RGB')); bs.append(dict(group=grp,cat=c,score=s))
            if c in ['cracked','honeycombed']:
                fig,ax=plt.subplots(1,2,figsize=(6,3)); ax[0].imshow(rec); ax[1].imshow(shf); [a.axis('off') for a in ax]; ax[0].set_title(c); ax[1].set_title('block-shuffled'); plt.savefig(f'{FIGS}/blockshuffle_{c}.png',dpi=150,bbox_inches='tight'); plt.show()
    json.dump(bs,open(BS,'w'),indent=1)
b=pd.DataFrame(bs); print(b.groupby('group').score.describe())
note('BlockShuffleStochastic', f"{b[b.group=='stochastic'].score.median():.2f}"); note('BlockShuffleRegular', f"{b[b.group=='regular'].score.median():.2f}")

In [ ]:
# ============================================================
# 6. FIGURES — all from this pipeline, seeded. Saved to FIGS/.
# ============================================================
GRID_PAIRS=[('honeycombed','cracked'),('marbled','scaly'),('fibrous','polka-dotted'),('banded','bubbly'),('woven','woven'),('lined','crystalline')]
def load_pair(ca,cb,s=7): return Image.open(dtd_path(ca,s)).convert('RGB'), Image.open(dtd_path(cb,s+50)).convert('RGB')
def mids(A,B,seed,etas=(0.5,)):
    d=build_d(cnt_load(A),cnt_load(B)); return {m:[to_pil(fn(d,e,seed=seed)) for e in etas] for m,fn in FNS.items()}, d

# 6a. MIDPOINT GRID (+ Texture Mixer column from saved n=1000 dir if the pair category matches; Vacher via stub)
cols=['A','pixel_linear','texton_linear','ot','ot_sym','B']
fig,ax=plt.subplots(len(GRID_PAIRS),len(cols),figsize=(2.2*len(cols),2.2*len(GRID_PAIRS)))
for r,(ca,cb) in enumerate(GRID_PAIRS):
    A,B=load_pair(ca,cb); M,_=mids(A,B,seed=r)
    imgs={'A':A.resize((256,256)),'B':B.resize((256,256)),**{m:M[m][0] for m in FNS}}
    for c,name in enumerate(cols):
        ax[r,c].imshow(imgs[name]); ax[r,c].axis('off')
        if r==0: ax[r,c].set_title(name.replace('_','-'))
    ax[r,0].set_ylabel(f'{ca}\n{cb}',rotation=0,labelpad=40,va='center')
plt.tight_layout(); plt.savefig(f'{FIGS}/grid_all_methods.png',dpi=150,bbox_inches='tight'); plt.show()
print('grid saved (add TM / Vacher columns offline from their saved midpoints)')

# 6b. ZOOM
A,B=load_pair('honeycombed','cracked'); M,_=mids(A,B,seed=0)
fig,ax=plt.subplots(2,2,figsize=(7,7))
for j,m in enumerate(['pixel_linear','ot']):
    im=M[m][0]; ax[0,j].imshow(im); ax[0,j].set_title(m.replace('_','-')); ax[1,j].imshow(im.crop((96,96,160,160)).resize((256,256),Image.NEAREST)); ax[1,j].set_title('64x64 crop, 4x')
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.savefig(f'{FIGS}/zoom_honeycomb.png',dpi=150,bbox_inches='tight'); plt.show()

# 6c. TRANSITION (2 pairs x 3 rows x 7 etas)
ETAS=[i/6 for i in range(7)]
for ca,cb in [('cracked','lined'),('marbled','scaly')]:
    A,B=load_pair(ca,cb); M,_=mids(A,B,seed=3,etas=ETAS)
    fig,ax=plt.subplots(3,7,figsize=(14,6))
    for r,m in enumerate(['pixel_linear','ot','ot_sym']):
        for c,e in enumerate(ETAS): ax[r,c].imshow(M[m][c]); ax[r,c].axis('off'); ax[r,c].set_title(f'η={e:.2f}' if r==0 else '')
        ax[r,0].text(-20,128,m.replace('_','-'),rotation=90,va='center',ha='right')
    plt.tight_layout(); plt.savefig(f'{FIGS}/transition_{ca}_{cb}.png',dpi=150,bbox_inches='tight'); plt.show()

# 6d. TEASER — structure-preserving transfer: eta=1 both directions
A,B=load_pair('honeycombed','marbled'); d=build_d(cnt_load(A),cnt_load(B)); dR=build_d(cnt_load(B),cnt_load(A))
t1=to_pil(interp_ot(d,1.0,seed=0)); t2=to_pil(interp_ot(dR,1.0,seed=0))
fig,ax=plt.subplots(1,4,figsize=(12,3))
for a,im,t in zip(ax,[A.resize((256,256)),t1,t2,B.resize((256,256))],['A','A layout, B material','B layout, A material','B']): a.imshow(im); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.savefig(f'{FIGS}/teaser_transfer.png',dpi=150,bbox_inches='tight'); plt.show()

# 6e. BOUNDARY — parametric rotations, crop pair, category pair, faces
def bandpass(theta_deg, f0=8, size=256, seed=0, bw=0.15):
    rng=np.random.RandomState(seed); n=rng.randn(size,size); Fq=np.fft.fftshift(np.fft.fft2(n))
    y,x=np.mgrid[-size//2:size//2,-size//2:size//2]; r=np.hypot(x,y)/size*256; th=np.arctan2(y,x)
    t=np.deg2rad(theta_deg); ang=np.cos(th-t)**8; rad=np.exp(-((np.log(r+1e-6)-np.log(f0))**2)/(2*bw**2))
    img=np.real(np.fft.ifft2(np.fft.ifftshift(Fq*ang*rad))); img=(img-img.mean())/(img.std()+1e-9)*0.2+0.5
    return Image.fromarray((np.clip(img,0,1)*255).astype(np.uint8)).convert('RGB')
FACE_A, FACE_B = None, None   # <-- set to two aligned face image paths (e.g. CelebA-HQ crops) to include row 5
rows=[('param 30°',bandpass(0),bandpass(30)),('param 70°',bandpass(0),bandpass(70))]
im=Image.open(dtd_path('cracked',9)).convert('RGB'); w,h=im.size; s=min(w,h); base=im.crop((0,0,s,s)); shift=int(0.25*s)
rows.append(('75% overlap crops',base.resize((256,256)),im.crop((shift,0,shift+s,s)).resize((256,256))))
rows.append(('different categories',*load_pair('bubbly','striped',11)))
if FACE_A and FACE_B: rows.append(('aligned faces',Image.open(FACE_A).convert('RGB').resize((256,256)),Image.open(FACE_B).convert('RGB').resize((256,256))))
fig,ax=plt.subplots(len(rows),4,figsize=(9,2.3*len(rows)))
for r,(name,A,B) in enumerate(rows):
    M,_=mids(A,B,seed=r+20)
    for c,(t,im) in enumerate([('A',A),('pixel-linear',M['pixel_linear'][0]),('OT',M['ot'][0]),('B',B)]):
        ax[r,c].imshow(im); ax[r,c].axis('off'); ax[r,c].set_title(t if r==0 else '')
    ax[r,0].set_ylabel(name,rotation=0,labelpad=60,va='center')
plt.tight_layout(); plt.savefig(f'{FIGS}/boundary.png',dpi=150,bbox_inches='tight'); plt.show()
print('figures written to', FIGS)

In [ ]:
# ============================================================
# 7. EMIT numbers_autofill.tex  — paste into numbers.tex (review each line; macro names match the paper's where they exist)
# ============================================================
alias={'FIDRefN':'FIDRefN','SubsampleM':'SubsampleM','SinkhornIters':'SinkhornIters','EncodeSec':'EncodeSec','EndpointGap':'EndpointGap',
       'RhoStarNScreened':'RhoStarNScreened','RhoStarHealthy':'RhoStarHealthy','RhoStarFail':'RhoStarFail','RowEntropyCeil':'RowEntropyCeil',
       'EpsEntropyHigh':'EpsEntropyHigh','EpsEntropyLow':'EpsEntropyLow','EpsVarLow':'EpsVarLow','EpsVarHigh':'EpsVarHigh',
       'Sec_ot_midpoint_total':'OTsec','Sec_ot_sym_midpoint_total':'OTsymSec','Sec_pixel_linear_midpoint_total':'LinearSec',
       'Sec_ot_path11_total':'PathSecOT','Sec_ot_sym_path11_total':'PathSecOTsym','Sec_pixel_linear_path11_total':'PathSecLinear',
       'Sec_tex_gram_midpoint':'VacherGrSec','Sec_tex_gauss_midpoint':'VacherGSec','Sec_tex_wasser_midpoint':'VacherWSec',
       'Sec_tex_gram_path11':'PathSecVacherGr','Sec_tex_gauss_path11':'PathSecVacherG','Sec_tex_wasser_path11':'PathSecVacherW',
       'Sec_texton_linear_midpoint_total':'TextonSec','Sec_texton_linear_path11_total':'PathSecTexton',
       'SpeedupW_midpoint':'VacherSpeedupW','SpeedupW_path':'VacherSpeedupWPath','SpeedupGr_midpoint':'VacherSpeedupGr','EpsDecodeDeltaLPIPS':'EpsDecodeDelta','RowEntropyDefault':'RowEntropy'}
with open(f'{OUT}/numbers_autofill.tex','w') as f:
    f.write('% ---- AUTOFILLED by paper_completion notebook ----\n')
    for k,v in NUM.items():
        name=alias.get(k,'AUTO'+k.replace('_',''))
        f.write(f'\\newcommand{{\\{name}}}{{{v}}}\n')
json.dump(NUM,open(f'{OUT}/numbers_autofill.json','w'),indent=1)
print(open(f'{OUT}/numbers_autofill.tex').read())

In [ ]:
# ============================================================
# 4b. rho* — SHARP and EXACT plans, all 1000 saved pairs. (rho at eps=0.05 is uninformative: plan ~uniform.)
# ============================================================
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
RH2=f'{OUT}/rhostar_1000_sharp_exact.csv'
EPS_SHARP=0.005; N_EXACT=1024

def rho_from_plan(P, xs, ys):
    xs=xs-xs.mean(0); ys=ys-ys.mean(0)
    return float((P*(xs@ys.T)).sum()/(xs.norm(dim=1).pow(2).mean().sqrt()*ys.norm(dim=1).pow(2).mean().sqrt()))
def rho_exact(xs, ys, n=N_EXACT, seed=0):
    g=torch.Generator(device=device).manual_seed(seed)
    i=torch.randperm(xs.shape[0],generator=g,device=device)[:n]; j=torch.randperm(ys.shape[0],generator=g,device=device)[:n]
    x=xs[i]-xs[i].mean(0); y=ys[j]-ys[j].mean(0)
    C=torch.cdist(x,y).pow(2).cpu().numpy(); r,c=linear_sum_assignment(C)
    xp=x[torch.as_tensor(r,device=device)]; yp=y[torch.as_tensor(c,device=device)]
    return float((xp*yp).sum(1).mean()/(x.norm(dim=1).pow(2).mean().sqrt()*y.norm(dim=1).pow(2).mean().sqrt()))

done = pd.read_csv(RH2) if os.path.exists(RH2) else pd.DataFrame(columns=['k','rho_sharp','rho_exact','rho_default'])
have=set(done.k.astype(int)); rows=done.to_dict('records')
MAN=json.load(open(f'{SCALE}/manifest.json'))
for p_ in MAN['pairs']:
    k=int(p_['k'])
    if k in have: continue
    A=Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'); B=Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB')
    d=build_d(cnt_load(A),cnt_load(B))
    _,info_s=interp_ot(d,0.5,seed=k,eps=EPS_SHARP,return_plan=True)
    _,info_d=interp_ot(d,0.5,seed=k,eps=EPS,return_plan=True)
    rows.append(dict(k=k,
        rho_sharp=rho_from_plan(info_s['P'],info_s['src'],info_s['tgt']),
        rho_default=rho_from_plan(info_d['P'],info_d['src'],info_d['tgt']),
        rho_exact=rho_exact(info_s['sf'],info_s['tf'],seed=k)))
    if k%25==0:
        pd.DataFrame(rows).to_csv(RH2,index=False); print(k, rows[-1])
rh=pd.DataFrame(rows).sort_values('k'); rh.to_csv(RH2,index=False)

for col in ['rho_exact','rho_sharp']:
    km=KMeans(2,n_init=10,random_state=0).fit(rh[[col]]); lo_c=np.argmin(km.cluster_centers_.ravel())
    flag=km.labels_==lo_c; hi=rh.loc[~flag,col]; lo=rh.loc[flag,col]
    print(f'\n{col}: healthy {hi.mean():.3f}±{hi.std():.3f} (n={len(hi)})  flagged {lo.mean():.3f}±{lo.std():.3f} (n={len(lo)})  '
          f'gap: min healthy {hi.min():.3f} / max flagged {lo.max():.3f}  -> {"SEPARATED" if hi.min()>lo.max() else "OVERLAP"}')
    if col=='rho_exact':
        rh['flag']=flag
        note('RhoStarNScreened',len(rh)); note('RhoStarHealthy',f'{hi.mean():.3f} \\pm {hi.std():.3f}')
        note('RhoStarFail',f'{lo.mean():.2f}'); note('RhoStarNFlagged',int(flag.sum()))
        note('RhoStarSeparated','yes' if hi.min()>lo.max() else 'no'); note('RhoStarAnomalyRate',f'{100*flag.mean():.1f}\\%')
print('corr exact vs sharp:', rh.rho_exact.corr(rh.rho_sharp), '  exact vs default:', rh.rho_exact.corr(rh.rho_default))
fig,ax=plt.subplots(1,3,figsize=(12,3))
for a,col in zip(ax,['rho_default','rho_sharp','rho_exact']): a.hist(rh[col],bins=60); a.set_title(col)
plt.savefig(f'{FIGS}/rhostar_hist_3.png',dpi=150,bbox_inches='tight'); plt.show()
print(rh[rh.flag].sort_values('rho_exact').head(15))